# Solution Lab: Multi-Variable Linear Regression — Feature Scaling, Learning Rate & NumPy Vectorization

Complete worked solution for the extended practice notebook.  All TODOs are filled; alternate implementations and the simulation are ready to run.


## Purpose of this Lab — What it Helps You Understand and Resolve

**Core purpose**  
This combined lab turns three related ideas into a single, practical workflow:

1. **NumPy vectorization** – replace slow Python loops with fast, readable `np.dot` / broadcasting operations that scale to real data sizes.
2. **Multi-variable linear regression** – extend the single-feature model to several house attributes (size, bedrooms, floors, age) so the model can make realistic price predictions.
3. **Feature scaling + learning-rate choice** – diagnose why gradient descent fails or crawls when feature magnitudes differ by orders of magnitude, and fix it with z-score normalization so a single, larger learning rate works for every parameter.

**Problems it resolves**
- “My gradient descent cost is *increasing*” → almost always an oversized learning rate relative to the raw feature scales.
- “Training takes forever / parameters move at very different speeds” → features with very different ranges (sqft vs number of bedrooms) produce gradients that differ by factors of 100–1000; scaling equalizes them.
- “I can predict only after I retrain everything” → you learn to *store* the training mean and standard deviation and apply the identical transform to any new house.
- “I don’t know which α to pick” → systematic trial of three regimes (too big, borderline, safe) plus the rule-of-thumb that after z-score scaling α ≈ 0.1 is a solid starting point.

**Audience value**
- **Practitioner / data scientist**: concrete, copy-pasteable routines for cost, gradient, and z-score that work with any tabular regression problem.
- **Engineer / technician**: clear diagnostics (cost curves, parameter oscillation plots) that tell you *why* the optimizer is misbehaving.
- **Decision maker**: a quantified demonstration that proper scaling turns a sluggish or divergent algorithm into a reliable pricing tool in a few hundred iterations.

By the end you will be able to take any multi-feature regression data set, scale it, run stable gradient descent, and produce a usable prediction pipeline.


## Analysis Flowchart

```mermaid
flowchart TD
    A[Load / generate housing data<br/>size, bedrooms, floors, age → price] --> B[Explore: scatter each feature vs price]
    B --> C[Implement vectorized predict / cost / gradient]
    C --> D{Try Gradient Descent<br/>without scaling}
    D -->|α too large| E[Cost diverges / oscillates]
    D -->|α too small| F[Cost decreases slowly]
    E --> G[Feature scaling: z-score normalize]
    F --> G
    G --> H[Re-run GD with larger α e.g. 0.1]
    H --> I[Fast convergence, accurate predictions]
    I --> J[Predict new house after normalizing with train μ,σ]
    J --> K[Simulation: vary α, noise, n_features]
```


## Detailed Cheat Sheet (keep open while working the skeleton)

### NumPy vectorization essentials
| Task | Code |
|------|------|
| Create vector | `a = np.array([1,2,3])` or `np.zeros(n)`, `np.arange(n)` |
| Dot product | `np.dot(a, b)`  (preferred over loop) |
| Column mean / std | `mu = X.mean(axis=0)`, `sigma = X.std(axis=0)` |
| Broadcast | `X_norm = (X - mu) / sigma` |
| Peak-to-peak | `np.ptp(X, axis=0)` |

### Multi-variable linear regression
$$
f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w}\cdot\mathbf{x} + b
$$
$$
J(\mathbf{w},b)=\frac{1}{2m}\sum_{i=0}^{m-1}(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)})^2
$$
$$
w_j := w_j - \alpha\frac{\partial J}{\partial w_j},\quad
b := b - \alpha\frac{\partial J}{\partial b}
$$
$$
\frac{\partial J}{\partial w_j}=\frac{1}{m}\sum_i(f-y)x_j^{(i)},\quad
\frac{\partial J}{\partial b}=\frac{1}{m}\sum_i(f-y)
$$

### Feature scaling (z-score)
$$
x_j^{(i)}\leftarrow\frac{x_j^{(i)}-\mu_j}{\sigma_j}
$$
- Store `mu`, `sigma` from **training** set; reuse for any new example.
- After scaling, learning rate can be much larger (e.g. `0.1`).

### Learning-rate diagnostics
| Symptom | Likely cause | Action |
|---------|--------------|--------|
| Cost **increases** | \(\alpha\) too large | Reduce \(\alpha\) by 3–10× |
| Cost decreases very slowly | \(\alpha\) too small or features unscaled | Scale features and/or increase \(\alpha\) |
| Oscillating parameters | \(\alpha\) near the stability limit | Slightly reduce \(\alpha\) |

### Audience adaptation (from supplied PDFs)
- **Data-literate / technician**: show equations, contour plots, gradient magnitudes.
- **Executive / decision maker**: lead with “scaled features cut training time dramatically and make the model usable for pricing new houses”.
- **Nonspecialist**: use concrete house example (1200 sqft, 3 bed…) and dollar prediction; avoid jargon.


## Goals (achieved)
- Vectorized multi-feature predict / cost / gradient
- Diagnosis of learning-rate failure modes on raw data
- Z-score scaling → fast, stable convergence with α = 0.1
- Correct out-of-sample prediction using stored μ, σ
- Alternates (min-max, fully vectorized gradient, sklearn)
- Parameterised Monte-Carlo-style simulation


## Step 1 — Imports


In [ ]:
import copy
import math
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
np.set_printoptions(precision=2, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
%matplotlib inline


## Step 2 — Load data


In [ ]:
df = pd.read_csv("data/housing_multi.csv")
print(df.shape)
print(df.head())
X_train = df[["size_sqft", "bedrooms", "floors", "age"]].values.astype(float)
y_train = df["price_1000s"].values.astype(float)
X_features = ["size_sqft", "bedrooms", "floors", "age"]
print("X shape:", X_train.shape, "y shape:", y_train.shape)


## Step 3 — Explore features vs price


In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for i in range(4):
    ax[i].scatter(X_train[:, i], y_train, alpha=0.7, edgecolor="k", linewidth=0.3)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("Price (1000s)")
fig.suptitle("Feature vs Price (raw data)")
plt.tight_layout()
plt.savefig("feature_vs_price.png", dpi=120, bbox_inches="tight")
plt.show()


## Step 4 — Prediction (loop + vectorized)


In [ ]:
def predict_loop(x, w, b):
    p = 0.0
    for j in range(len(x)):
        p += w[j] * x[j]
    return p + b

def predict(x, w, b):
    return np.dot(x, w) + b

w_demo = np.array([0.1, 10.0, -5.0, -1.0])
b_demo = 50.0
x0 = X_train[0]
print("loop :", predict_loop(x0, w_demo, b_demo))
print("vector:", predict(x0, w_demo, b_demo))


## Step 5 — Cost


In [ ]:
def compute_cost(X, y, w, b):
    m = X.shape[0]
    cost = 0.0
    for i in range(m):
        f = np.dot(X[i], w) + b
        cost += (f - y[i]) ** 2
    return cost / (2 * m)

print("Cost at zero weights:", compute_cost(X_train, y_train, np.zeros(4), 0.0))


## Step 6 — Gradient (loop version)


In [ ]:
def compute_gradient(X, y, w, b):
    m, n = X.shape
    dj_dw = np.zeros(n)
    dj_db = 0.0
    for i in range(m):
        err = (np.dot(X[i], w) + b) - y[i]
        for j in range(n):
            dj_dw[j] += err * X[i, j]
        dj_db += err
    return dj_db / m, dj_dw / m

dj_db, dj_dw = compute_gradient(X_train, y_train, np.zeros(4), 0.0)
print("dj_db:", dj_db)
print("dj_dw:", dj_dw)


## Step 7 — Gradient descent driver


In [ ]:
def gradient_descent(X, y, w_in, b_in, alpha, num_iters,
                     cost_fn=compute_cost, grad_fn=compute_gradient, verbose=True):
    w = copy.deepcopy(w_in)
    b = b_in
    J_hist = []
    for i in range(num_iters):
        dj_db, dj_dw = grad_fn(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        if i < 100000:
            J_hist.append(cost_fn(X, y, w, b))
        if verbose and i % max(1, num_iters // 10) == 0:
            print(f"Iter {i:4d}: cost {J_hist[-1]:.4f}")
    return w, b, J_hist


## Step 8 — Learning-rate experiments (raw features)


In [ ]:
alphas = [9.9e-7, 9e-7, 1e-7]
histories = {}
for a in alphas:
    print(f"\n=== alpha = {a} ===")
    w0 = np.zeros(X_train.shape[1])
    _, _, hist = gradient_descent(X_train, y_train, w0, 0.0, a, 30, verbose=True)
    histories[a] = hist

fig, ax = plt.subplots(1, 3, figsize=(14, 3))
for i, a in enumerate(alphas):
    ax[i].plot(histories[a], marker="o", markersize=3)
    ax[i].set_title(f"α = {a}")
    ax[i].set_xlabel("iteration")
    ax[i].set_ylabel("cost")
plt.tight_layout()
plt.savefig("learning_rate_raw.png", dpi=120, bbox_inches="tight")
plt.show()


**Interpretation (audience notes)**  
- Technician: the 9.9e-7 run shows classic overshoot — cost rises because the step size exceeds the curvature of the cost surface along the size-feature direction.  
- Executive: without scaling, the optimizer either diverges or needs thousands of tiny steps; the model is not production-ready.


## Step 9 — Z-score normalization


In [ ]:
def zscore_normalize(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    sigma = np.where(sigma == 0, 1.0, sigma)
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

X_norm, mu, sigma = zscore_normalize(X_train)
print("mu   :", mu)
print("sigma:", sigma)
print("ptp raw :", np.ptp(X_train, axis=0))
print("ptp norm:", np.ptp(X_norm, axis=0))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(X_train[:, 0], X_train[:, 3], alpha=0.7)
ax[0].set_xlabel("size_sqft"); ax[0].set_ylabel("age")
ax[0].set_title("Raw (unequal scales)")
ax[0].axis("equal")
ax[1].scatter(X_norm[:, 0], X_norm[:, 3], alpha=0.7)
ax[1].set_xlabel("size (z)"); ax[1].set_ylabel("age (z)")
ax[1].set_title("Z-score normalized")
ax[1].axis("equal")
plt.tight_layout()
plt.savefig("zscore_effect.png", dpi=120, bbox_inches="tight")
plt.show()


## Step 10 — GD on scaled features (α = 0.1)


In [ ]:
w_norm, b_norm, hist_norm = gradient_descent(
    X_norm, y_train, np.zeros(4), 0.0, alpha=0.1, num_iters=1000
)
print("Final w:", w_norm)
print("Final b:", b_norm)
print("Final cost:", hist_norm[-1])

plt.figure(figsize=(6, 3))
plt.plot(hist_norm)
plt.xlabel("iteration"); plt.ylabel("cost")
plt.title("Cost vs iteration (scaled features, α=0.1)")
plt.savefig("cost_scaled.png", dpi=120, bbox_inches="tight")
plt.show()


## Step 11 — Predictions vs targets


In [ ]:
yp = X_norm @ w_norm + b_norm

fig, ax = plt.subplots(1, 4, figsize=(14, 3), sharey=True)
for i in range(4):
    ax[i].scatter(X_train[:, i], y_train, label="target", alpha=0.6)
    ax[i].scatter(X_train[:, i], yp, label="predict", alpha=0.6)
    ax[i].set_xlabel(X_features[i])
ax[0].set_ylabel("Price (1000s)")
ax[0].legend()
fig.suptitle("Target vs prediction (z-score model)")
plt.tight_layout()
plt.savefig("pred_vs_target.png", dpi=120, bbox_inches="tight")
plt.show()


## Step 12 — New-house prediction


In [ ]:
x_new = np.array([1200., 3., 1., 40.])
x_new_norm = (x_new - mu) / sigma
price_pred = np.dot(x_new_norm, w_norm) + b_norm
print(f"Predicted price of 1200 sqft / 3 bed / 1 floor / 40 yr: ${price_pred * 1000:,.0f}")


## Alternate A — Min-max scaling


In [ ]:
def minmax_normalize(X):
    X_min = X.min(axis=0)
    X_max = X.max(axis=0)
    rng = np.where(X_max == X_min, 1.0, X_max - X_min)
    return (X - X_min) / rng, X_min, X_max

X_mm, xmin, xmax = minmax_normalize(X_train)
w_mm, b_mm, hist_mm = gradient_descent(X_mm, y_train, np.zeros(4), 0.0, 0.1, 800, verbose=False)
print("minmax final cost:", hist_mm[-1])
print("z-score final cost:", hist_norm[-1])


## Alternate B — Fully vectorized gradient


In [ ]:
def compute_gradient_vectorized(X, y, w, b):
    m = X.shape[0]
    err = (X @ w + b) - y
    dj_dw = (X.T @ err) / m
    dj_db = err.sum() / m
    return dj_db, dj_dw

d1, dw1 = compute_gradient(X_norm, y_train, w_norm, b_norm)
d2, dw2 = compute_gradient_vectorized(X_norm, y_train, w_norm, b_norm)
print("db match:", np.isclose(d1, d2))
print("dw match:", np.allclose(dw1, dw2))


## Alternate C — sklearn StandardScaler


In [ ]:
scaler = StandardScaler()
X_sk = scaler.fit_transform(X_train)
print("sklearn mean_:", scaler.mean_)
print("manual mu    :", mu)
print("Close?", np.allclose(scaler.mean_, mu))


## More Practice (solutions)


In [ ]:
# 1. Long unscaled run with tiny alpha
_, _, hist_long = gradient_descent(
    X_train, y_train, np.zeros(4), 0.0, 1e-8, 3000, verbose=False
)
print("Unscaled 3000 iter final cost:", hist_long[-1])
print("Scaled 1000 iter final cost  :", hist_norm[-1])

# 3. Mean-normalization
def mean_normalize(X):
    mu = X.mean(axis=0)
    rng = X.max(axis=0) - X.min(axis=0)
    rng = np.where(rng == 0, 1.0, rng)
    return (X - mu) / rng, mu, rng

X_mn, mu_mn, rng_mn = mean_normalize(X_train)
w_mn, b_mn, hist_mn = gradient_descent(X_mn, y_train, np.zeros(4), 0.0, 0.1, 800, verbose=False)
print("Mean-norm final cost:", hist_mn[-1])


## Simulation Section


In [ ]:
# === SIMULATION CONTROLS ===
SIM_ALPHA      = 0.1
SIM_ITERS      = 500
SIM_NOISE_STD  = 40.0
SIM_N_SAMPLES  = 100
SIM_SEED       = 123
# ===========================

rng = np.random.default_rng(SIM_SEED)
size = rng.integers(800, 3500, SIM_N_SAMPLES)
beds = rng.integers(1, 6, SIM_N_SAMPLES)
floors = rng.integers(1, 3, SIM_N_SAMPLES)
age = rng.integers(5, 80, SIM_N_SAMPLES)
price = (50 + 0.12*size + 15*beds - 20*floors - 1.2*age
         + rng.normal(0, SIM_NOISE_STD, SIM_N_SAMPLES))
X_sim = np.column_stack([size, beds, floors, age]).astype(float)
y_sim = np.clip(price, 80, 900)

X_sim_n, mu_s, sig_s = zscore_normalize(X_sim)
w_s, b_s, hist_s = gradient_descent(
    X_sim_n, y_sim, np.zeros(4), 0.0, SIM_ALPHA, SIM_ITERS, verbose=False
)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(hist_s)
ax[0].set_title(f"Cost (α={SIM_ALPHA}, noise={SIM_NOISE_STD})")
ax[0].set_xlabel("iteration")
ax[1].scatter(y_sim, X_sim_n @ w_s + b_s, alpha=0.7)
lims = [y_sim.min(), y_sim.max()]
ax[1].plot(lims, lims, "r--")
ax[1].set_xlabel("true price"); ax[1].set_ylabel("predicted")
ax[1].set_title("Prediction quality")
plt.tight_layout()
plt.savefig("simulation_results.png", dpi=120, bbox_inches="tight")
plt.show()
print("Final cost:", hist_s[-1])
print("w:", w_s, "b:", b_s)


## Key Takeaways
- Divergent or oscillating cost on raw data is almost always a learning-rate / feature-scale mismatch.
- Z-score (or min-max) scaling lets you use a single, aggressive learning rate (≈ 0.1) and reach a good minimum in a few hundred iterations.
- Store training μ and σ; any new example must be transformed with the same statistics before prediction.
- Prefer vectorized NumPy (`X @ w`, `X.T @ err`) once shapes are clear — faster and less error-prone.
- Cost-vs-iteration plots are the fastest diagnostic for α and scaling problems.
